In [1]:
import time

import h3
import pandas as pd

BASE = "https://cct-ds-code-challenge-input-data.s3.af-south-1.amazonaws.com/"

sr = pd.read_csv(
    BASE + "sr.csv.gz",
    index_col=0,
    dtype={"notification_number": str, "reference_number": str},
)


def to_hex(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return "0"
    return h3.latlng_to_cell(lat, lon, 8)


start = time.perf_counter()
sr["h3_level8_index"] = [to_hex(lat, lon) for lat, lon in zip(sr["latitude"], sr["longitude"])]
print(f"assigned hexes in {time.perf_counter() - start:.1f}s")

print((sr["h3_level8_index"] == "0").sum(), "requests with hex 0")
sr[["latitude", "longitude", "h3_level8_index"]].head()

assigned hexes in 1.5s
212364 requests with hex 0


,latitude,longitude,h3_level8_index
0,-33.872839,18.522488,88ad360225fffff
1,-34.078916,18.848940,88ad36d5e1fffff
2,-34.102242,18.821116,88ad36d437fffff
3,-33.920019,18.607209,88ad361133fffff
4,-33.987400,18.453760,88ad361709fffff


In [ ]:
import geopandas as gpd

hex8 = gpd.read_file("../data/hex8.geojson") #loads section 1 output
city_hexes = set(hex8["index"])

has_location = sr["h3_level8_index"] != "0"
failed = has_location & ~sr["h3_level8_index"].isin(city_hexes) # not in city list

print(has_location.sum(), "requests with a location")
print(failed.sum(), "join failures")
print(sr.loc[failed, ["notification_number", "official_suburb", "latitude", "longitude", "h3_level8_index"]])

print("\nnotification_number unique:", sr["notification_number"].is_unique) #checks requests are unique

ref = pd.read_csv(
    BASE + "sr_hex.csv.gz",
    usecols=["notification_number", "h3_level8_index"],
    dtype=str,
)

#ref is from city sr_hex, _ours is from hex calculated with h3 in section 1
compare = sr[["notification_number", "h3_level8_index"]].merge(
    ref,
    on="notification_number",
    how="left",
    suffixes=("_ours", "_ref"),
    validate="one_to_one",
)

matches = compare["h3_level8_index_ours"] == compare["h3_level8_index_ref"]
print(matches.sum(), "of", len(compare), "match sr_hex")
print(compare[~matches].head())

729270 requests with a location
3 join failures
       notification_number            official_suburb   latitude  longitude  \
462434        001015950287                        NaN -34.044257  18.774378   
804983        001016348681                        NaN -34.044257  18.774378   
924595        001016488950  BOTTELARY SMALLHOLDINGS 1 -33.904955  18.723060   

        h3_level8_index  
462434  88ad36c629fffff  
804983  88ad36c629fffff  
924595  88ad361b51fffff  

notification_number unique: True
941634 of 941634 match sr_hex
Empty DataFrame
Columns: [notification_number, h3_level8_index_ours, h3_level8_index_ref]
Index: []


In [3]:
swapped = pd.Series(
    [to_hex(lon, lat) for lat, lon in zip(sr["latitude"], sr["longitude"])],
    index=sr.index,
)
swapped_failed = (swapped != "0") & ~swapped.isin(city_hexes)
print(f"{swapped_failed.sum() / has_location.sum():.1%} fail when lat/lon are swapped")

100.0% fail when lat/lon are swapped
